# **GPU 스펙 확인**

In [ ]:
!nvidia-smi

# **라이브러리추가**

In [ ]:
import os
import random
import cv2

# Tensorflow 관련 디버그 및 경고 메시지 비활성화 (삭제 금지)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
import tensorflow.image as tfi
from tensorflow.keras import Sequential
from tensorflow.keras import layers, models
from tensorflow.keras import backend as K

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image, ImageDraw
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# **폴더 경로 설정**

In [ ]:
data_path = '/kaggle/input/competitions/2026-DAU-CV'

# **재구현 세팅**

In [ ]:
def init_seeds(seed):
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    np.random.seed(seed)
    random.seed(seed)

In [ ]:
init_seeds(2026)

# **데이터 로드**

In [ ]:
train_image_path = os.path.join(data_path, 'train/images')
train_label_path = os.path.join(data_path, 'train/masks')
test_image_path = os.path.join(data_path, 'test/images')

output_path = '/kaggle/working'

In [ ]:
train_images = os.listdir(train_image_path)
train_images = [os.path.join(train_image_path, x) for x in train_images]
train_labels = os.listdir(train_label_path)
train_labels = [os.path.join(train_label_path, x) for x in train_labels]

train_images.sort(), train_labels.sort()

test_images = os.listdir(test_image_path)
test_images = [os.path.join(test_image_path, x) for x in test_images]

test_images.sort()

# **이미지 시각화**

In [ ]:
sample_img = Image.open(train_images[0]).convert("RGB")
sample_label = Image.open(train_labels[0]).convert("L")

print(f'sample image size : {sample_img.size}, sample label size : {sample_label.size}')

img_np = np.array(sample_img)
label_np = np.array(sample_label)

label_color = cm.jet(label_np / 255.0)[:, :, :3] 

overlay = (0.7 * img_np / 255.0 + 0.3 * label_color)
overlay = np.clip(overlay, 0, 1)

fig = plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_np)
plt.title("Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(label_color)
plt.title("Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

# **데이터 전처리**

In [ ]:
def build_train_dataset(image_paths, mask_paths, img_height=256, img_width=256):
    X, y = [], []

    for img_path, mask_path in tqdm(zip(image_paths, mask_paths), total=len(image_paths)):

        # MRI 이미지를 RGB로 읽고 모델 입력 크기에 맞게 resize 후 0~1 범위로 정규화
        img = cv2.imread(img_path)                
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) 
        img = cv2.resize(img, (img_width, img_height))
        img = img.astype(np.float32) / 255.0     
        X.append(img)

        # 정답 mask는 grayscale로 읽은 뒤 이진 mask(0 또는 1)로 변환
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)  
        mask = cv2.resize(mask, (img_width, img_height))
        mask = (mask > 127).astype(np.float32)           
        mask = np.expand_dims(mask, axis=-1)             
        y.append(mask)

    X = np.stack(X, axis=0)
    y = np.stack(y, axis=0)
    print(f"Dataset loaded: X={X.shape}, y={y.shape}")
    return X, y


def build_test_dataset(image_paths, img_height=256, img_width=256):
    X = []
    for img_path in tqdm(image_paths):
        # test 이미지는 mask가 없으므로 학습 이미지와 동일한 방식으로만 전처리
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_width, img_height))
        img = img.astype(np.float32) / 255.0
        X.append(img)

    X = np.stack(X, axis=0)
    print(f"Test dataset loaded: X={X.shape}")
    return X

In [ ]:
img_height, img_width = 256, 256
train_X, train_y = build_train_dataset(train_images, train_labels, img_height = img_height, img_width = img_width)
test_X = build_test_dataset(test_images, img_height = img_height, img_width = img_width)

In [ ]:
# 학습 데이터의 10%를 검증 데이터로 분리하여 학습 중 일반화 성능을 확인
train_X_split, val_X_split, train_y_split, val_y_split = train_test_split(
    train_X, train_y, test_size=0.1, random_state=2026, shuffle=True
)

print("train:", train_X_split.shape, train_y_split.shape)
print("valid:", val_X_split.shape, val_y_split.shape)

In [ ]:
# plt.imshow(train_y[0]) # 인덱스 번호를 바꾸면 다른 학습 mask 확인 가능 (필요시 주석 제거)

In [ ]:
# 데이터 증강 설정
# 1) 이미지와 mask에 동일한 기하학적 변환을 적용하여 위치 불일치 방지
# 2) 밝기/대비/gamma 변형은 이미지에만 약하게 적용하여 MRI 밝기 차이에 대응
# 3) mask는 증강 후에도 항상 0/1 이진값으로 유지

AUTO = tf.data.AUTOTUNE

SEED = 2026

def augment_image_mask(image, mask):
    image = tf.cast(image, tf.float32)
    mask = tf.cast(mask, tf.float32)

    # 좌우 반전
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # 상하 반전
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    # 90도 단위 회전
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    # 이미지에만 약한 intensity augmentation 적용
    image = tf.image.random_brightness(image, max_delta=0.08)
    image = tf.image.random_contrast(image, lower=0.92, upper=1.12)

    # 약한 gamma 변형
    gamma = tf.random.uniform([], 0.90, 1.10)
    image = tf.pow(tf.clip_by_value(image, 0.0, 1.0), gamma)

    image = tf.clip_by_value(image, 0.0, 1.0)
    mask = tf.cast(mask > 0.5, tf.float32)

    return image, mask


def make_dataset(images, masks=None, batch_size=8, training=False):
    if masks is not None:
        ds = tf.data.Dataset.from_tensor_slices((images, masks))
        if training:
            ds = ds.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
            ds = ds.map(augment_image_mask, num_parallel_calls=AUTO)
        ds = ds.batch(batch_size).prefetch(AUTO)
    else:
        ds = tf.data.Dataset.from_tensor_slices(images)
        ds = ds.batch(batch_size).prefetch(AUTO)
    return ds

# **모델 정의**

In [ ]:
# 모델 구조 정의
# 1) MobileNetV2 encoder 기반 U-Net 구조 사용
# 2) bottleneck에 ASPP를 추가하여 다양한 크기의 종양 영역을 더 잘 반영
# 3) skip connection에 Spatial Attention을 적용하여 종양 관련 영역에 집중
# 4) Conv2DTranspose 대신 UpSampling2D를 사용하여 checkerboard artifact 위험 완화

def conv_bn_relu(x, filters, kernel_size=3, dilation_rate=1):
    x = tf.keras.layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False,
        kernel_initializer="he_normal"
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    return x


class ChannelMean(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)


class ChannelMax(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)


def spatial_attention(x):
    avg_pool = ChannelMean()(x)
    max_pool = ChannelMax()(x)

    concat = tf.keras.layers.Concatenate(axis=-1)([avg_pool, max_pool])

    attention = tf.keras.layers.Conv2D(
        1,
        kernel_size=7,
        padding="same",
        activation="sigmoid"
    )(concat)

    return tf.keras.layers.Multiply()([x, attention])


def aspp_block(x, filters=256):
    shape = tf.keras.backend.int_shape(x)

    y1 = conv_bn_relu(x, filters, kernel_size=1, dilation_rate=1)
    y2 = conv_bn_relu(x, filters, kernel_size=3, dilation_rate=2)
    y3 = conv_bn_relu(x, filters, kernel_size=3, dilation_rate=4)
    y4 = conv_bn_relu(x, filters, kernel_size=3, dilation_rate=6)

    # image-level feature
    y5 = tf.keras.layers.GlobalAveragePooling2D()(x)
    y5 = tf.keras.layers.Reshape((1, 1, shape[-1]))(y5)
    y5 = conv_bn_relu(y5, filters, kernel_size=1)
    y5 = tf.keras.layers.UpSampling2D(size=(shape[1], shape[2]), interpolation="bilinear")(y5)

    y = tf.keras.layers.Concatenate()([y1, y2, y3, y4, y5])
    y = conv_bn_relu(y, filters, kernel_size=1)
    return y


def build_model(input_shape=(256, 256, 3)):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )

    # pretrained encoder의 초반 저수준 feature는 고정하고, 이후 layer는 fine-tuning
    for layer in base_model.layers[:60]:
        layer.trainable = False
    for layer in base_model.layers[60:]:
        layer.trainable = True

    skip_layer_names = [
        "block_1_expand_relu",   # 128x128
        "block_3_expand_relu",   # 64x64
        "block_6_expand_relu",   # 32x32
        "block_13_expand_relu",  # 16x16
    ]
    skip_outputs = [base_model.get_layer(name).output for name in skip_layer_names]
    encoder_output = base_model.get_layer("block_16_project").output  # 8x8
    encoder = tf.keras.Model(inputs=base_model.input, outputs=skip_outputs + [encoder_output])

    inputs = tf.keras.Input(shape=input_shape)
    x_in = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)

    s1, s2, s3, s4, x = encoder(x_in)

    # bottleneck에서 multi-scale feature 추출
    x = aspp_block(x, filters=256)

    def decoder_block(x, skip, filters, dropout_rate=0.0):
        x = tf.keras.layers.UpSampling2D((2, 2), interpolation="bilinear")(x)
        skip = spatial_attention(skip)
        x = tf.keras.layers.Concatenate()([x, skip])
        x = conv_bn_relu(x, filters, 3)
        if dropout_rate > 0:
            x = tf.keras.layers.SpatialDropout2D(dropout_rate)(x)
        x = conv_bn_relu(x, filters, 3)
        return x

    x = decoder_block(x, s4, 320, dropout_rate=0.15)
    x = decoder_block(x, s3, 160, dropout_rate=0.10)
    x = decoder_block(x, s2, 80, dropout_rate=0.05)
    x = decoder_block(x, s1, 40, dropout_rate=0.03)

    x = tf.keras.layers.UpSampling2D((2, 2), interpolation="bilinear")(x)
    x = conv_bn_relu(x, 32, 3)
    x = tf.keras.layers.SpatialDropout2D(0.02)(x)
    outputs = tf.keras.layers.Conv2D(1, 1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)
    return model

# **손실 함수 정의**

In [ ]:
# 손실 함수 및 평가 지표 정의
# 1) BCE + Dice + Focal loss를 조합하여 픽셀 단위 정확도와 종양 영역 겹침을 함께 반영
# 2) Dice loss는 mask overlap을 높이는 데 사용
# 3) Focal loss는 작은 종양이나 어려운 픽셀을 놓치는 문제를 줄이기 위해 낮은 비중으로 추가

def dice_loss(y_true, y_pred, smooth=1e-5):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    denominator = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return 1.0 - tf.reduce_mean(dice)


def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)

    bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
    alpha_factor = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
    modulating_factor = tf.pow(1.0 - p_t, gamma)

    return tf.reduce_mean(alpha_factor * modulating_factor * bce)


def combo_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce)

    dloss = dice_loss(y_true, y_pred)
    floss = focal_loss(y_true, y_pred)

    return 0.40 * bce + 0.45 * dloss + 0.15 * floss


def dice_coefficient(y_true, y_pred, smooth=1e-5):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    denominator = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return tf.reduce_mean(dice)


def iou_coefficient(y_true, y_pred, smooth=1e-5):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2, 3]) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return tf.reduce_mean(iou)


# **모델 생성 및 컴파일**

In [ ]:
# 모델 생성 및 컴파일
# 1) AdamW optimizer 사용: weight decay로 과적합 완화
# 2) learning rate=1e-4, weight_decay=1e-5 설정
# 3) val_dice_coefficient가 가장 높은 모델을 best_model_v3.keras로 저장

model = build_model(input_shape=(256, 256, 3))

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=1e-4,
    weight_decay=1e-5
)

model.compile(
    optimizer=optimizer,
    loss=combo_loss,
    metrics=[dice_coefficient, iou_coefficient]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model_v3.keras",
        monitor="val_dice_coefficient",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_dice_coefficient",
        mode="max",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",
        mode="max",
        patience=7,
        restore_best_weights=True,
        verbose=1
    )
]


# **모델 학습**

In [ ]:
# 모델 학습
# 최대 50 epoch까지 학습하되, EarlyStopping과 ModelCheckpoint로 검증 성능이 가장 좋은 모델을 사용

BATCH_SIZE = 8
EPOCHS = 50

train_ds = make_dataset(train_X_split, train_y_split, batch_size=BATCH_SIZE, training=True)
val_ds = make_dataset(val_X_split, val_y_split, batch_size=BATCH_SIZE, training=False)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

model = tf.keras.models.load_model(
    "best_model_v3.keras",
    custom_objects={
        "ChannelMean": ChannelMean,
        "ChannelMax": ChannelMax
    },
    compile=False
)


In [ ]:
#예측값 확인 예시 (필요시 주석 제거)
# pred = model.predict(train_X[:5])
# print(pred.min(), pred.max())
# plt.hist(pred.flatten(), bins=50)
# plt.title("Predicted Probability Distribution")
# plt.xlabel("Predicted Value")
# plt.ylabel("Pixel Count")
# plt.show()

# **모델 예측**

In [ ]:
# Test Time Augmentation(TTA)
# 학습에서 사용한 flip/rot90 변환과 맞춰 test 예측도 여러 방향으로 수행한 뒤 평균
# 마지막 pred ** 0.82 보정은 예측 확률을 약간 높여 mask가 지나치게 작아지는 것을 완화

def tta_predict(model, test_images, batch_size=8):
    preds = []

    # original
    pred = model.predict(make_dataset(test_images, batch_size=batch_size, training=False), verbose=1)
    preds.append(pred)

    # horizontal flip
    imgs = np.flip(test_images, axis=2).copy()
    pred = model.predict(make_dataset(imgs, batch_size=batch_size, training=False), verbose=1)
    preds.append(np.flip(pred, axis=2))

    # vertical flip
    imgs = np.flip(test_images, axis=1).copy()
    pred = model.predict(make_dataset(imgs, batch_size=batch_size, training=False), verbose=1)
    preds.append(np.flip(pred, axis=1))

    # horizontal + vertical flip
    imgs = np.flip(np.flip(test_images, axis=1), axis=2).copy()
    pred = model.predict(make_dataset(imgs, batch_size=batch_size, training=False), verbose=1)
    preds.append(np.flip(np.flip(pred, axis=1), axis=2))

    # rot90 TTA
    for k in [1, 2, 3]:
        imgs = np.rot90(test_images, k=k, axes=(1, 2)).copy()
        pred = model.predict(make_dataset(imgs, batch_size=batch_size, training=False), verbose=1)
        pred = np.rot90(pred, k=-k, axes=(1, 2))
        preds.append(pred)

    pred_mean = np.mean(preds, axis=0)
    return pred_mean

pred = tta_predict(model, test_X, batch_size=BATCH_SIZE)
pred = pred ** 0.82


In [ ]:
# 예측결과 확인 (필요시 주석 제거)
plt.imshow(pred[0])

# **Run-Length Encoding 및 예측 결과 저장**

In [ ]:
output_path = '/kaggle/working'
result_path = os.path.join(output_path, 'results')
os.makedirs(result_path, exist_ok=True)

In [ ]:
# 수정X
def rle_encode(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    if len(runs) % 2 != 0:
        runs = np.append(runs, len(pixels) - 1)
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

In [ ]:
# 수정X

THRESHOLD = 0.5

rle_list = []

for i in range(pred.shape[0]):
    prob = pred[i, :, :, 0]
    mask = (prob > THRESHOLD).astype(np.uint8)

    rle = rle_encode(mask)
    rle_list.append(rle)

    mask_img = mask * 255
    cv2.imwrite(os.path.join(result_path, f"{i+1:04d}.jpg"), mask_img)

# **제출 CSV 파일 저장**

In [ ]:
submission = pd.read_csv(os.path.join(data_path, 'sample_submission.csv'))

In [ ]:
submission['EncodedPixels'] = rle_list
submission.head()

In [ ]:
submission.to_csv('sample_submission.csv', index=False)

In [ ]:
import shutil
shutil.make_archive('label', 'zip', result_path)

In [ ]:
shutil.rmtree(result_path)